# Which tokens IterGen's grammar step rejects

**Runtime: G4 GPU.** Then **Runtime > Run all**. Nothing to edit. About 15-20 minutes.

Replays IterGen (checks off) on the tasks that unconstrained decoding passes and IterGen fails,
and prints every token IterGen judged invalid (`tools/itergen_rejections.py`). Output:
`final/itergen_rejections_<commit>/` on Drive. The runtime disconnects itself at the end.

In [ ]:
REPO_URL    = 'https://github.com/BowenX307/constrained-llm-js-codegen-final.git'
REPO_COMMIT = '0d2fc0c9cc6274fd2004f347c1af50a5f0ca2ef4'
EXPECT_GPU  = 'RTX PRO 6000'
TASKS = {
    'Qwen/Qwen2.5-3B-Instruct': ['mbpp_230_replace_blank', 'mbpp_285_text_match_two_three', 'mbpp_604_reverse_words',
                                 'mbpp_628_replace_spaces', 'mbpp_757_count_reverse_pairs', 'mbpp_771_check_expression',
                                 'mbpp_772_remove_length', 'mbpp_787_text_match_three'],
    'Qwen/Qwen3.5-4B': ['mbpp_230_replace_blank', 'mbpp_285_text_match_two_three', 'mbpp_565_split',
                        'mbpp_619_move_num', 'mbpp_757_count_reverse_pairs'],
}
SOURCE_RUN = 'itergen_off_2c14eef'
AUTO_DISCONNECT = True

In [ ]:
import os, sys, shutil, subprocess, time
REPO, CACHE = '/content/final', '/content/cache'
DRIVE_ROOT = '/content/drive/MyDrive/constrained-llm-js-codegen'
MS_DRIVE = f'{DRIVE_ROOT}/mask_store_zst'
os.makedirs(CACHE, exist_ok=True)
smi = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
GPU_NAME = smi.stdout.strip().splitlines()[0] if smi.returncode == 0 and smi.stdout.strip() else ''
print('GPU', GPU_NAME or 'none')
if EXPECT_GPU not in GPU_NAME:
    raise RuntimeError(f'This runtime has {GPU_NAME or "no GPU"}; use G4 (Runtime > Change runtime type).')
from google.colab import drive
drive.mount('/content/drive')
if os.path.isdir(REPO):
    shutil.rmtree(REPO)
for cmd in (['git', 'init', '-q', REPO], ['git', '-C', REPO, 'remote', 'add', 'origin', REPO_URL],
            ['git', '-C', REPO, 'fetch', '-q', '--depth', '1', 'origin', REPO_COMMIT],
            ['git', '-C', REPO, 'checkout', '-q', 'FETCH_HEAD']):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(' '.join(cmd) + ' failed:\n' + r.stderr)
def pip(*a):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('pip install failed: ' + ' '.join(a) + '\n' + r.stderr[-2000:])
from importlib.metadata import version
if version('transformers') != '5.20.1':
    was = version('transformers')
    pip('transformers==5.20.1')
    if 'transformers' in sys.modules:
        raise RuntimeError(f'transformers was {was}; installed 5.20.1. Runtime > Restart session, then Run all.')
pip('lark', 'regex', 'networkx', 'fire', 'interegular', 'tqdm', 'datasets', 'jsonschema', 'psutil')
if subprocess.run([sys.executable, '-c', 'from syncode.parsers.itergen_parser import IGParser'],
                  cwd=REPO, capture_output=True).returncode != 0:
    pip('--no-deps', '--ignore-requires-python', '--force-reinstall',
        'git+https://github.com/structuredllm/syncode.git@4d6c110a028d3fb9cbb5f1a6ac6446b696f3eac4')
if not shutil.which('zstd'):
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], capture_output=True)
ENV = dict(os.environ, SYNCODE_CACHE=CACHE + '/syncode/', HF_HOME=CACHE + '/hf_home', PYTHONPATH=REPO,
           PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_PROGRESS_BARS='1')
print('ready, code at', REPO_COMMIT[:7])

In [ ]:
def restore_mask_store(model):
    grammar = f'{REPO}/generators/grammars/javascript.lark'
    probe = subprocess.run([sys.executable, '-c',
        'import syncode.common as c; from syncode.parsers.grammars import Grammar\n'
        f'tok = c.load_tokenizer({model!r}); g = Grammar({grammar!r})\n'
        'print("MS_PATH=" + c.SYNCODE_CACHE + "mask_stores/" + type(tok).__name__ + '
        '"/grammar_strict_" + g.hash() + "_" + str(tok.vocab_size) + ".pkl")'],
        env=ENV, cwd=REPO, capture_output=True, text=True)
    lines = [l for l in probe.stdout.splitlines() if l.startswith('MS_PATH=')]
    if probe.returncode != 0 or not lines:
        raise RuntimeError('could not resolve the mask store path:\n' + probe.stderr[-2000:])
    path = lines[-1][len('MS_PATH='):]
    zst = f'{MS_DRIVE}/{os.path.basename(path)}.zst'
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not (os.path.exists(path) and os.path.getsize(path) > 100 * 2**20):
        if not os.path.exists(zst):
            raise RuntimeError(f'mask store not on Drive: {zst}')
        r = subprocess.run(['zstd', '-q', '-d', '-f', '-T0', zst, '-o', path + '.part'], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError('the copy on Drive is unreadable: ' + r.stderr[-300:])
        os.replace(path + '.part', path)
    print('mask store ready:', os.path.basename(path))

OUT_DRIVE = f'{DRIVE_ROOT}/final/itergen_rejections_{REPO_COMMIT[:7]}'
os.makedirs(OUT_DRIVE, exist_ok=True)
try:
    for model, names in TASKS.items():
        slug = model.split('/')[-1]
        print(f'== {slug}', flush=True)
        restore_mask_store(model)
        src = f'/content/work/{slug}/{SOURCE_RUN}'
        os.makedirs(src, exist_ok=True)
        for n in names:
            shutil.copy(f'{DRIVE_ROOT}/final/{slug}/{SOURCE_RUN}/{n}.json', src)
        out = f'/content/work/{slug}_itergen_rejections.jsonl'
        r = subprocess.run([sys.executable, f'{REPO}/tools/itergen_rejections.py', '--model', model, '--run', src,
                            '--out', out, *names], cwd=REPO, env=ENV, text=True,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print('\n'.join(l for l in r.stdout.splitlines()
                        if not any(s in l for s in ('Warning', 'warn(', 'Loading weights', 'torch_dtype', 'httpx'))))
        if r.returncode != 0:
            raise RuntimeError(f'{slug}: itergen_rejections.py failed')
        shutil.copy(out, OUT_DRIVE)
        print('saved to', OUT_DRIVE, flush=True)
    print('ALL DONE')
finally:
    if AUTO_DISCONNECT:
        from google.colab import drive, runtime
        drive.flush_and_unmount()
        time.sleep(30)
        runtime.unassign()